# BEVFusion KD Training Notebook
**Dissertation: Knowledge-Distilled BEV Perception for ADAS**

Run cells **top to bottom**. Each cell checks if it already completed so re-running is safe.

| Cell | What it does | Time |
|------|-------------|------|
| 0 | Mount Drive, check GPU + nuScenes | 30s |
| 1 | Clone repo, install dependencies | 2min |
| 2 | Verify student model loads | 10s |
| 3 | **Baseline 20-epoch training** | ~3-4h |
| 4 | Resume baseline if disconnected | — |
| 5 | Attempt BEVFusion teacher install | 5min |
| 6 | Generate teacher cache | 5-45min |
| 7 | **KD 20-epoch training** | ~3-4h |
| 8 | Resume KD if disconnected | — |
| 9 | Results summary table | 5s |

**Before starting:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# CELL 0: Mount Drive + verify GPU + check nuScenes
from google.colab import drive
drive.mount('/content/drive')

import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DRIVE_ROOT    = '/content/drive/MyDrive/dissertation-bev'
NUSCENES_PATH = f'{DRIVE_ROOT}/data/nuscenes'
CACHE_PATH    = f'{DRIVE_ROOT}/data/teacher_cache'
CKPT_PATH     = f'{DRIVE_ROOT}/checkpoints'
LOGS_PATH     = f'{DRIVE_ROOT}/logs'

for p in [CACHE_PATH, CKPT_PATH, LOGS_PATH]:
    os.makedirs(p, exist_ok=True)

print('\n--- nuScenes check ---')
if os.path.exists(NUSCENES_PATH):
    for item in sorted(os.listdir(NUSCENES_PATH)):
        item_path = os.path.join(NUSCENES_PATH, item)
        if os.path.isdir(item_path):
            n = sum(len(f) for _, _, f in os.walk(item_path))
            print(f'  OK {item}/  ({n} files)')
    print('\nOK nuScenes found')
else:
    print(f'MISSING: {NUSCENES_PATH}')

In [ ]:
# CELL 1: Clone repo + install student dependencies
import os

REPO_URL = 'https://github.com/neomond/lightweight-bev-adas.git'
REPO_DIR = '/content/repo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')
    print('Repo already exists, pulled latest')

os.chdir(REPO_DIR)

# Symlink data directories
os.makedirs('data', exist_ok=True)
for local, remote in [
    ('data/nuscenes',      NUSCENES_PATH),
    ('data/teacher_cache', CACHE_PATH),
    ('checkpoints',        CKPT_PATH),
    ('logs',               LOGS_PATH),
]:
    if not os.path.exists(local) and os.path.exists(remote):
        os.symlink(remote, local)
        print(f'Symlinked {local}')

os.system('pip install -q nuscenes-devkit pyquaternion ultralytics tensorboard tqdm pyyaml')
print('Dependencies installed')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# CELL 2: Verify student model loads
import sys, os
sys.path.insert(0, '/content/repo')
os.chdir('/content/repo')

import torch, yaml
from src.models.student import StudentBEV

with open('configs/student.yaml') as f:
    config = yaml.safe_load(f)

model = StudentBEV(config['model'])
params = model.count_parameters()
print('Student model parameters:')
for name, count in params.items():
    print(f'  {name:<30} {count:>10,}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'\nModel loaded on {device}')

In [ ]:
# CELL 3: PHASE 1 - Full 20-epoch baseline training (no teacher needed)
# ~3-4 hours on T4. Checkpoints save to Drive every epoch.
import os, torch
from pathlib import Path
os.chdir('/content/repo')

baseline_best = Path(CKPT_PATH) / 'baseline_20ep_best.pth'
if baseline_best.exists():
    ckpt = torch.load(baseline_best, map_location='cpu', weights_only=False)
    print(f'Baseline already trained: val_loss={ckpt["val_loss"]:.4f} epoch={ckpt["epoch"]}')
    print('Skip to Cell 5.')
else:
    print('Starting 20-epoch baseline training...')
    os.system('python scripts/train_baseline.py '
              '--config configs/student.yaml '
              '--epochs 20 --batch 4 --workers 4 '
              '--run-name baseline_20ep')

In [ ]:
# CELL 4: Resume baseline if Colab disconnected (only run if Cell 3 was interrupted)
import os, torch
from pathlib import Path
os.chdir('/content/repo')

latest = Path(CKPT_PATH) / 'baseline_20ep_latest.pth'
if latest.exists():
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'Resuming baseline from epoch {ckpt["epoch"]}')
    os.system('python scripts/train_baseline.py '
              '--config configs/student.yaml '
              '--epochs 20 --batch 4 --workers 4 '
              '--run-name baseline_20ep')
else:
    print('No checkpoint found. Run Cell 3 first.')

In [ ]:
# CELL 5: Attempt BEVFusion teacher install
# Tries mmdet3d compatible with CUDA 12.x.
# If it fails, Cell 6 uses mock teacher automatically.
import subprocess, sys

print('Attempting mmdet3d install...')

# Try pip direct install of mmdet3d v1.4 (supports CUDA 12)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'mmdet3d'],
    capture_output=True, text=True, timeout=300
)

try:
    import importlib
    mmdet3d = importlib.import_module('mmdet3d')
    print(f'mmdet3d available: {getattr(mmdet3d, "__version__", "unknown version")}')
    TEACHER_MODE = 'real'
except Exception as e:
    print(f'mmdet3d not available: {e}')
    print('Will use mock teacher cache in Cell 6')
    TEACHER_MODE = 'mock'

print(f'Teacher mode: {TEACHER_MODE}')

In [ ]:
# CELL 6: Generate teacher cache
# Auto-detects real vs mock teacher. Saves to Drive. Only needs to run once.
import os
from pathlib import Path
os.chdir('/content/repo')

train_dir = Path(CACHE_PATH) / 'train'
val_dir   = Path(CACHE_PATH) / 'val'
train_count = len(list(train_dir.glob('*.pt'))) if train_dir.exists() else 0
val_count   = len(list(val_dir.glob('*.pt')))   if val_dir.exists()   else 0
print(f'Existing cache: {train_count} train, {val_count} val')

if train_count >= 324 and val_count >= 80:
    print('Cache complete. Skip to Cell 7.')
else:
    try:
        import mmdet3d
        mock_flag = ''
        print('Using real BEVFusion teacher')
    except ImportError:
        mock_flag = '--mock'
        print('Using mock teacher')

    for split in ['train', 'val']:
        print(f'Caching {split}...')
        os.system(f'python scripts/cache_teacher_outputs.py '
                  f'--config configs/student.yaml {mock_flag} '
                  f'--split {split} --cache-dir data/teacher_cache')

    train_count = len(list(train_dir.glob('*.pt')))
    val_count   = len(list(val_dir.glob('*.pt')))
    print(f'Cache complete: {train_count} train, {val_count} val')

In [ ]:
# CELL 7: PHASE 3 - Full 20-epoch KD training
# ~3-4 hours on T4. Checkpoints save to Drive every epoch.
import os, torch
from pathlib import Path
os.chdir('/content/repo')

kd_best = Path(CKPT_PATH) / 'kd_real_best.pth'
if kd_best.exists():
    ckpt = torch.load(kd_best, map_location='cpu', weights_only=False)
    print(f'KD already trained: val_loss={ckpt["val_loss"]:.4f} epoch={ckpt["epoch"]}')
else:
    print('Starting 20-epoch KD training...')
    os.system('python scripts/train_with_kd.py '
              '--config configs/student.yaml '
              '--epochs 20 --batch 4 --workers 4 '
              '--run-name kd_real '
              '--cache-dir data/teacher_cache')

In [ ]:
# CELL 8: Resume KD training if disconnected (only run if Cell 7 was interrupted)
import os, torch
from pathlib import Path
os.chdir('/content/repo')

latest = Path(CKPT_PATH) / 'kd_real_latest.pth'
if latest.exists():
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'Resuming KD from epoch {ckpt["epoch"]}')
    os.system('python scripts/train_with_kd.py '
              '--config configs/student.yaml '
              '--epochs 20 --batch 4 --workers 4 '
              f'--run-name kd_real '
              '--cache-dir data/teacher_cache '
              f'--resume {CKPT_PATH}/kd_real_latest.pth')
else:
    print('No KD checkpoint found. Run Cell 7 first.')

In [ ]:
# CELL 9: Results summary table
import torch
from pathlib import Path

print('=' * 55)
print('  DISSERTATION RESULTS TABLE')
print('=' * 55)

checkpoints = {
    'Baseline (no KD), 20ep':  f'{CKPT_PATH}/baseline_20ep_best.pth',
    'Student + KD, 20ep':      f'{CKPT_PATH}/kd_real_best.pth',
}

results = {}
for name, path in checkpoints.items():
    if Path(path).exists():
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        results[name] = ckpt['val_loss']
        print(f'  {name:<30} val_loss={ckpt["val_loss"]:.4f}  epoch={ckpt["epoch"]}')
    else:
        print(f'  {name:<30} NOT YET TRAINED')

if len(results) == 2:
    k = list(results.keys())
    delta = results[k[0]] - results[k[1]]
    pct   = delta / results[k[0]] * 100
    print()
    print(f'  KD improvement: {delta:+.4f}  ({pct:.1f}% val loss reduction)')
    if delta > 0:
        print('  KD improved over baseline')
    else:
        print('  KD did not improve (expected with mock teacher)')
print('=' * 55)